In [2]:
from ultralytics import YOLO
yolo_seg_model=YOLO("yolov8n-seg")

In [3]:
yolo_seg_model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,

    scale=0.5,        
    translate=0.1,
    degrees=5.0,
    shear=2.0,

    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,

   
    mosaic=0.5,
    mixup=0.0,         
)

New https://pypi.org/project/ultralytics/8.4.4 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.243  Python-3.9.25 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.6, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=0.5, multi_scale=False, name=train, nbs=64, nms=False, opset=Non

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002A532D93400>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.0

In [4]:
metrics =yolo_seg_model.val(
    data='C:\Hope AI\course\Deep Learning\DL Project\Project 9 Yolo-Seg\data.yaml', 
    split='test',            # Loading the test set
    imgsz=640,               # training image size
    batch=8,                # GPU memory
    conf=0.3,              # confidence for mAP calculation
    iou=0.6,save=True                 # IoU threshold for NMS
)

# 3. Print the results
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

Ultralytics 8.3.243  Python-3.9.25 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLOv8n-seg summary (fused): 85 layers, 3,258,454 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access  (ping: 0.40.0 ms, read: 47.521.9 MB/s, size: 23.5 KB)
val: Scanning C:\Hope AI\course\Deep Learning\DL Project\Project 9 Yolo-Seg\test\labels.cache... 71 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 71/71  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 2.6it/s 3.4s0.2ss
                   all         71         78      0.966      0.975      0.989      0.968      0.957      0.966      0.984      0.937
                  silo         21         21      0.933          1      0.995      0.979      0.933          1      0.995      0.926
                vessel         56         57          1       0.95      0.982      0.957      0.982      0.933      0.972    

In [6]:
import cv2
import os
from ultralytics import YOLO
import mediapipe as mp
from datetime import datetime

# ===================== CONFIG =====================
VIDEO_PATH = r"milk_collecting.mp4"
MODEL_PATH = r"runs/segment/train/weights/best.pt"

BASE_SAVE_DIR = r"theft_frames"

CONF_THRES = 0.5
THEFT_FRAME_THRESHOLD = 8
CAPTURE_COOLDOWN = 20
FRAMES_TO_SAVE = 3   # number of frames to save per event

os.makedirs(BASE_SAVE_DIR, exist_ok=True)

# ===================== MODELS =====================
yolo = YOLO(MODEL_PATH)

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ===================== VIDEO =====================
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), "❌ Cannot open video"

# Read first frame to get size
ret, frame = cap.read()
assert ret, "❌ Cannot read first frame"

H, W = frame.shape[:2]
fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:
    fps = 25

# Reset video to start
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

# ===================== STATE =====================
theft_counter = 0
cooldown = 0
frame_id = 0

current_event_dir = None
saved_frames = 0

# ===================== PROCESS =====================
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1

    results = yolo(frame, conf=CONF_THRES, verbose=False)[0]
    vis_frame = results.plot()

    silo_box = None
    vessel_box = None

    for box in results.boxes:
        cls = int(box.cls[0])
        label = yolo.names[cls]
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

        if label == "silo":
            silo_box = (x1, y1, x2, y2)
        elif label == "vessel":
            vessel_box = (x1, y1, x2, y2)

    status = "CLEAR"

    if silo_box and vessel_box:
        # ---------- SILO MID ----------
        sx1, sy1, sx2, sy2 = silo_box
        silo_mid_y = (sy1 + sy2) / 2

        cv2.line(vis_frame, (0, int(silo_mid_y)), (W, int(silo_mid_y)),
                 (0, 255, 255), 2)

        # ---------- VESSEL ----------
        vx1, vy1, vx2, vy2 = vessel_box
        vessel_center_y = (vy1 + vy2) / 2
        vessel_above = vessel_center_y < silo_mid_y

        cv2.circle(vis_frame,
                   (int((vx1 + vx2) / 2), int(vessel_center_y)),
                   5, (0, 0, 255), -1)

        # ---------- MEDIAPIPE ----------
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pose_result = pose.process(rgb)

        human_above = False
        if pose_result.pose_landmarks:
            lm = pose_result.pose_landmarks.landmark
            hip_y = ((lm[mp_pose.PoseLandmark.LEFT_HIP].y +
                      lm[mp_pose.PoseLandmark.RIGHT_HIP].y) / 2) * H

            cv2.circle(vis_frame, (W // 2, int(hip_y)),
                       6, (255, 0, 255), -1)

            human_above = hip_y <= silo_mid_y

        # ---------- TEMPORAL ----------
        if vessel_above and human_above:
            theft_counter += 1
            status = f"THEFT CANDIDATE ({theft_counter})"
        else:
            theft_counter = 0
            current_event_dir = None
            saved_frames = 0

        # ---------- CONFIRMED EVENT ----------
        if theft_counter == THEFT_FRAME_THRESHOLD and cooldown == 0:
            timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            current_event_dir = os.path.join(BASE_SAVE_DIR, timestamp)
            os.makedirs(current_event_dir, exist_ok=True)

            saved_frames = 0
            cooldown = CAPTURE_COOLDOWN
            status = "THEFT CAPTURED"

        # ---------- SAVE FRAMES ----------
        if current_event_dir and saved_frames < FRAMES_TO_SAVE:
            save_path = os.path.join(
                current_event_dir, f"frame_{frame_id}.jpg"
            )
            cv2.imwrite(save_path, vis_frame)
            saved_frames += 1

    if cooldown > 0:
        cooldown -= 1

    # ---------- STATUS TEXT ----------
    color = (0, 255, 0)
    if "CANDIDATE" in status:
        color = (0, 255, 255)
    elif "CAPTURED" in status:
        color = (0, 0, 255)

    cv2.putText(vis_frame, status, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

    # ---------- DISPLAY ----------
    cv2.imshow("Milk Theft Detection", vis_frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

# ===================== CLEANUP =====================
cap.release()
pose.close()
cv2.destroyAllWindows()

print("✅ Theft frame saving completed.")

✅ Theft frame saving completed.
